# COSMoS Graph — Query Cookbook

Seven minimal queries against the schema-driven flatten. Each query is pinned
to a DSS already documented in the HTML story pairs under `docs/`, so the same
worked examples appear in three formats:

| Structural sub-type         | Story DSS   | HTML story                             |
|-----------------------------|-------------|----------------------------------------|
| Specimen-based Finding (LB) | `GLUCPL`    | `docs/Glucose_COSMoS_Story.html`       |
| Instrument Finding (FT)     | `SIXMW101`  | `docs/6MWT_COSMoS_Story.html`          |
| Measurement Finding (MK)    | `SGBESCR`   | `docs/XRay_COSMoS_Story.html`          |

**Inputs** — `cosmos-bc-dss/interim/COSMoS_Graph.xlsx` (notebook 10 output)
and `cosmos-bc-dss/interim/COSMoS_Graph_CT.xlsx` (notebook 20 output).

**What this proves** — the graph shape carries enough signal to answer the
questions the infographics pose narratively. No new columns, no fabrication.


## Setup

Load both workbooks. All cookbook queries reuse these DataFrames.


In [ ]:
from pathlib import Path
import pandas as pd

REPO = Path.cwd()
while not (REPO / 'cosmos-bc-dss').exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / 'cosmos-bc-dss').exists(), f'cosmos-bc-dss not found under {Path.cwd()}'

GRAPH    = REPO / 'cosmos-bc-dss' / 'interim' / 'COSMoS_Graph.xlsx'
GRAPH_CT = REPO / 'cosmos-bc-dss' / 'interim' / 'COSMoS_Graph_CT.xlsx'

dss_df       = pd.read_excel(GRAPH, 'DSS')
variables_df = pd.read_excel(GRAPH, 'Variables')
rel_df       = pd.read_excel(GRAPH, 'Relationships')
codelists_df = pd.read_excel(GRAPH, 'Codelists')
bc_df        = pd.read_excel(GRAPH, 'BC')

codelist_terms_df = pd.read_excel(GRAPH_CT, 'CodelistTerms')
assigned_terms_df = pd.read_excel(GRAPH_CT, 'AssignedTerms')

print('DSS            :', dss_df.shape)
print('Variables      :', variables_df.shape)
print('Relationships  :', rel_df.shape)
print('Codelists      :', codelists_df.shape)
print('BC             :', bc_df.shape)
print('CodelistTerms  :', codelist_terms_df.shape)
print('AssignedTerms  :', assigned_terms_df.shape)

STORY_DSS = ['GLUCPL', 'SIXMW101', 'SGBESCR']


## Q1 — Identity pins: TESTCD, TEST, BC

What identity does the DSS pin? A DSS is authored as a value-level metadata
group, so the *pins* are the `Variables` rows with a non-null
`assigned_term_value`. For a Findings DSS, the Topic pin is the TESTCD
assignment; the paired Qualifier pin (via the `decodes the value in`
relationship) is the TEST label.

**Story parallel.** Glucose_COSMoS_Story.html box "VLM group GLUCPL — 12 rows",
6MWT_COSMoS_Story.html SIXMW101 table, XRay_COSMoS_Story.html "What SGBESCR
pins vs leaves open".


In [ ]:
def identity_pins(ds_id: str) -> pd.DataFrame:
    """Return the assigned-term pins for a DSS, joined to DSS and BC for context."""
    pins = variables_df[
        (variables_df['ds_id'] == ds_id)
        & variables_df['assigned_term_value'].notna()
    ][['variable_name', 'role', 'assigned_term_value', 'assigned_term_concept_id']]
    return pins.reset_index(drop=True)

for ds in STORY_DSS:
    dss_row = dss_df[dss_df['ds_id'] == ds].iloc[0]
    bc_row  = bc_df[bc_df['bc_id'] == dss_row['bc_id']].iloc[0]
    print(f'\n=== {ds}  ({dss_row["domain"]})  on {dss_row["source"]}  ===')
    print(f'  BC    : {bc_row["bc_short_name"]}  ({bc_row["bc_id"]})')
    print(f'  DSS   : {dss_row["ds_short_name"]}')
    print('  Pins :')
    print(identity_pins(ds).to_string(index=False))


## Q2 — Specimen (specimen-based Findings only)

For specimen-based Findings the specimen is modelled as a pinned qualifier
with its own reification edge. Find it via the relationship with
`predicate_term = 'IS_SPECIMEN_TESTED_IN'`, then join back to `Variables`
to read the pinned value.

**Story parallel.** Glucose_COSMoS_Story.html — the case-defining specimen
attribute is what distinguishes `GLUCPL` from `GLUCBLD` from `GLUCSER`.


In [ ]:
def specimen_for_dss(ds_id: str) -> pd.DataFrame:
    """Return any specimen pin for a DSS, via the IS_SPECIMEN_TESTED_IN edge."""
    spec_edges = rel_df[
        (rel_df['ds_id'] == ds_id)
        & (rel_df['predicate_term'] == 'IS_SPECIMEN_TESTED_IN')
    ]
    if spec_edges.empty:
        return pd.DataFrame(columns=['variable_name', 'assigned_term_value',
                                     'assigned_term_concept_id',
                                     'codelist_submission_value'])
    return variables_df.merge(
        spec_edges[['ds_id', 'variable_name']], on=['ds_id', 'variable_name']
    )[['variable_name', 'assigned_term_value', 'assigned_term_concept_id',
       'codelist_submission_value']].reset_index(drop=True)

for ds in STORY_DSS:
    print(f'\n=== {ds}  specimen ===')
    spec = specimen_for_dss(ds)
    if spec.empty:
        print('  (no specimen edge — not a specimen-based Finding)')
    else:
        print(spec.to_string(index=False))


## Q3 — Standard result unit

The standard-unit slot is the variable that participates in an
`IS_UNIT_FOR` edge whose `object` is the numeric standard result
variable (`xxSTRESN`). Read the pin (`assigned_term_value`) and
the bound codelist. For DSSs that leave the unit open, the pin is
absent and `value_list` carries the permitted set.

**Story parallel.** Glucose_COSMoS_Story.html flattened-row section shows
LBSTRESU = mmol/L; 6MWT story shows FTSTRESU = m uniformly.


In [ ]:
def standard_unit_for_dss(ds_id: str) -> pd.DataFrame:
    """Return standard-unit pin and bound codelist for a DSS via IS_UNIT_FOR edge."""
    unit_edges = rel_df[
        (rel_df['ds_id'] == ds_id)
        & (rel_df['predicate_term'] == 'IS_UNIT_FOR')
        & (rel_df['object'].str.endswith('STRESN', na=False))
    ]
    if unit_edges.empty:
        return pd.DataFrame()
    return variables_df.merge(
        unit_edges[['ds_id', 'variable_name']], on=['ds_id', 'variable_name']
    )[['variable_name', 'assigned_term_value', 'assigned_term_concept_id',
       'codelist_submission_value', 'codelist_concept_id', 'value_list']]\
     .reset_index(drop=True)

for ds in STORY_DSS:
    print(f'\n=== {ds}  standard result unit ===')
    u = standard_unit_for_dss(ds)
    if u.empty:
        print('  (no IS_UNIT_FOR xxSTRESN edge)')
    else:
        print(u.to_string(index=False))


## Q4 — Method: pinned vs value_list

Method in MK can be either pinned (one term assigned) or left open
(`value_list` carries the permitted modalities). The `xxMETHOD`
variable name is the simplest selector; the shape to report is
(`assigned_term_value`, `value_list`).

**Story parallel.** XRay_COSMoS_Story.html — SGBESCR leaves `MKMETHOD` open
with `X-RAY;MRI`. One DSS covers two modalities.


In [ ]:
def method_for_dss(ds_id: str) -> pd.DataFrame:
    rows = variables_df[
        (variables_df['ds_id'] == ds_id)
        & variables_df['variable_name'].str.endswith('METHOD', na=False)
    ]
    return rows[['variable_name', 'assigned_term_value', 'assigned_term_concept_id',
                 'codelist_submission_value', 'codelist_concept_id',
                 'value_list']].reset_index(drop=True)

for ds in STORY_DSS:
    print(f'\n=== {ds}  method ===')
    m = method_for_dss(ds)
    if m.empty:
        print('  (no xxMETHOD variable on this DSS)')
    else:
        print(m.to_string(index=False))


## Q5 — Permissible values for a bound codelist

When a DSS binds a codelist (not a `value_list` inline subset), the
permissible values live in SDTM CT, not in the DSS. Look them up in
`CodelistTerms` (from `COSMoS_Graph_CT.xlsx`).

Worked example: `LBSPEC` on `GLUCPL` binds the `SPECTYPE` codelist
(C78734). Query pulls all permissible terms in that codelist context.


In [ ]:
def codelist_members(codelist_concept_id: str) -> pd.DataFrame:
    members = codelist_terms_df[
        codelist_terms_df['codelist_concept_id'] == codelist_concept_id
    ][['term_concept_id', 'term_submission_value', 'term_nci_preferred_term']]\
    .reset_index(drop=True)
    return members

# GLUCPL.LBSPEC binds SPECTYPE (C78734)
glucpl_spec = variables_df[
    (variables_df['ds_id'] == 'GLUCPL')
    & (variables_df['variable_name'] == 'LBSPEC')
][['codelist_submission_value', 'codelist_concept_id',
   'assigned_term_value', 'assigned_term_concept_id']]
print('GLUCPL.LBSPEC binding:')
print(glucpl_spec.to_string(index=False))

spec_terms = codelist_members('C78734')
print(f'\nSPECTYPE (C78734) — {len(spec_terms)} permissible terms (first 15):')
print(spec_terms.head(15).to_string(index=False))


## Q6 — BC parent and hierarchy

The classification layer: which BC does this DSS sit under, and where
in the hierarchy. `bc_type` distinguishes `full` (composition +
classification) from `hierarchy_only` (classification shell).

**Story parallel.** Glucose story notes thin classification (one BC,
many DSSs). 6MWT story notes rich classification with a dedicated
parent `6MWT Functional Test Question`. XRay sits on the measurement
concept C128986.


In [ ]:
def bc_for_dss(ds_id: str) -> pd.DataFrame:
    dss_row = dss_df[dss_df['ds_id'] == ds_id].iloc[0]
    bc_row = bc_df[bc_df['bc_id'] == dss_row['bc_id']].iloc[0]
    return pd.DataFrame([{
        'ds_id':            ds_id,
        'bc_id':            bc_row['bc_id'],
        'bc_short_name':    bc_row['bc_short_name'],
        'bc_type':          bc_row['bc_type'],
        'bc_parent_label':  bc_row['bc_parent_label'],
        'bc_hierarchy_path': bc_row['bc_hierarchy_path'],
    }])

for ds in STORY_DSS:
    row = bc_for_dss(ds).iloc[0]
    print(f'\n=== {ds} ===')
    for k, v in row.items():
        print(f'  {k:20s}: {v}')


## Q7 — Composite DSS identity card

Assemble Q1–Q6 into a single card per DSS. This is the row-shape the
eventual consumer tracks (`sdtm-findings/`) will materialise —
Test_Identity + Measurement_Specs. The cookbook proves the signal
needed to populate those files is already in the graph.


In [ ]:
def identity_card(ds_id: str) -> dict:
    dss_row = dss_df[dss_df['ds_id'] == ds_id].iloc[0]
    bc_row  = bc_df[bc_df['bc_id'] == dss_row['bc_id']].iloc[0]

    pins = identity_pins(ds_id)
    topic = pins[pins['role'] == 'Topic']
    testcd = topic['assigned_term_value'].iloc[0] if not topic.empty else None
    # TEST = qualifier pin with same assigned_term_concept_id as topic
    test_label = None
    if not topic.empty:
        topic_cid = topic['assigned_term_concept_id'].iloc[0]
        q = pins[
            (pins['role'] == 'Qualifier')
            & (pins['assigned_term_concept_id'] == topic_cid)
            & (pins['variable_name'].str.endswith('TEST', na=False))
        ]
        if not q.empty:
            test_label = q['assigned_term_value'].iloc[0]

    spec = specimen_for_dss(ds_id)
    unit = standard_unit_for_dss(ds_id)
    method = method_for_dss(ds_id)

    def first(df, col):
        return df[col].iloc[0] if not df.empty else None

    def first_non_null(df, col):
        if df.empty:
            return None
        s = df[col].dropna()
        return s.iloc[0] if not s.empty else None

    return {
        'ds_id':       ds_id,
        'domain':      dss_row['domain'],
        'source':      dss_row['source'],
        'bc':          f'{bc_row["bc_short_name"]}  ({bc_row["bc_id"]})',
        'bc_type':     bc_row['bc_type'],
        'testcd':      testcd,
        'test':        test_label,
        'specimen':    first(spec, 'assigned_term_value'),
        'unit_pin':    first(unit, 'assigned_term_value'),
        'unit_value_list': first_non_null(unit, 'value_list'),
        'method_pin':      first(method, 'assigned_term_value'),
        'method_value_list': first_non_null(method, 'value_list'),
    }

cards = pd.DataFrame([identity_card(ds) for ds in STORY_DSS])
print(cards.T.to_string())


## Q8 — X-Ray cross-domain-class concept traversal

One NCIt concept, two structural roles. `C38101 — X-Ray` appears in the
graph two ways:

- As an **identity pin** on Interventions-style DSSs (PR) where the DSS
  *is* the X-Ray procedure — surfaces as
  `Variables.assigned_term_concept_id == 'C38101'`.
- As a **permissible codelist value** on Findings-style DSSs (MK) where
  the DSS leaves method open and X-Ray is one term in the bound
  `MKMETHOD` codelist — surfaces as a `CodelistTerms` row, reachable
  via the codelist that `Variables.codelist_concept_id` binds.

One concept, two roles. A one-hop traversal should recover both.

Two codelists carry `C38101` in the 2026-Q1 graph:
`PROCEDUR` (C101858), bound only on PR; and `METHOD` (C85492), bound on
12 domains (FA, GF, IS, LB, MB, MI, MK, SR, TR, TU, UR, VS). `METHOD` is
the same codelist every time — reused, not re-authored per domain — so
the per-domain row count in the result table reflects how many DSSs in
that domain have an `xxMETHOD` variable, not anything specific about
X-Ray. The signal is the *binding*, not the count.

The structural split surfaces cleanly. Interventions-style (PR) models
the procedure as pinned identity on `xxDECOD`/`xxTRT`; Findings-style
(12 domains) models the method as a qualifier slot bound to a shared
CT codelist. Q8 recovers both in one traversal.

**Story parallel.** XRay_COSMoS_Story.html — the cross-domain-class
section that motivated this rewrite.


In [ ]:
XRAY_CID = 'C38101'

# Branch 1 — identity pins: Variables rows where X-Ray is the assigned term
identity_hits = variables_df[
    variables_df['assigned_term_concept_id'] == XRAY_CID
][['ds_id', 'variable_name', 'role',
   'assigned_term_value', 'codelist_submission_value']].copy()
identity_hits['role_in_dss'] = 'identity_pin'
identity_hits = identity_hits.rename(columns={'assigned_term_value': 'value'})
identity_hits = identity_hits[['ds_id', 'variable_name', 'role',
                               'role_in_dss', 'value',
                               'codelist_submission_value']]

# Branch 2 — codelist value: codelists that contain X-Ray, joined to Variables
codelists_with_xray = codelist_terms_df[
    codelist_terms_df['term_concept_id'] == XRAY_CID
][['codelist_concept_id', 'term_submission_value']].drop_duplicates()

codelist_hits = variables_df.merge(
    codelists_with_xray, on='codelist_concept_id'
)[['ds_id', 'variable_name', 'role',
   'codelist_submission_value', 'term_submission_value']].copy()
codelist_hits['role_in_dss'] = 'codelist_value'
codelist_hits = codelist_hits.rename(columns={'term_submission_value': 'value'})
codelist_hits = codelist_hits[['ds_id', 'variable_name', 'role',
                               'role_in_dss', 'value',
                               'codelist_submission_value']]

traversal = pd.concat([identity_hits, codelist_hits], ignore_index=True)

# Enrich with DSS domain + source for context
traversal = traversal.merge(
    dss_df[['ds_id', 'domain', 'source', 'ds_short_name']],
    on='ds_id', how='left'
)[['ds_id', 'domain', 'source', 'ds_short_name',
   'variable_name', 'role', 'role_in_dss', 'value',
   'codelist_submission_value']]

n_pin = int((traversal['role_in_dss'] == 'identity_pin').sum())
n_val = int((traversal['role_in_dss'] == 'codelist_value').sum())
print(f'C38101 (X-Ray) traversal — {len(traversal)} hits '
      f'({n_pin} identity pins, {n_val} codelist values)')
print()
print('By role_in_dss x domain:')
print(traversal.groupby(['role_in_dss', 'domain']).size().to_string())
print()
print('Detail (first 30 rows):')
print(traversal.head(30).to_string(index=False))


## Summary

Seven queries, three DSSs, one graph. Each HTML story has a companion
query here; each query returns non-empty output grounded in the real
authored graph. Once this notebook is stable, the HTML stories can be
refreshed so that every claim in the infographic has an executable
query behind it.

Next steps (tracked separately):

- Port the consumer tracks (`sdtm-findings/Specimen_Findings.xlsx`, etc.)
  to read from this graph rather than the legacy `COSMoS_BC_DSS.xlsx`.
- Refresh the three HTML stories to show the new schema-driven shape,
  citing the cookbook queries.
